In [1]:
### Preamples ###
import numpy as np
import matplotlib.pyplot as plt
import random
from numba import njit

random.seed(13)

In [2]:
@njit
def heston_returns(V0, N, mu, theta, kappa, sigma_V, rho, h, n_sub):
    
    dt = h / n_sub
    sqrt_dt = dt ** 0.5
    c = (1 - rho ** 2) ** 0.5 
    
    returns = np.empty(N)
  
    Vt = V0
    log_ret = 0
    
    for n in range(N):
        log_ret = 0
        for _ in range(n_sub):
            z1 = np.random.normal(0, 1)
            z2 = np.random.normal(0, 1)
            
            dB1 = sqrt_dt * z1
            dB2 = sqrt_dt * (rho * z1 + c * z2) 
            
            Vt_pos = Vt if Vt > 0 else 0 # ensure variance is non-negative for the price update
            
            log_ret += (mu - 0.5 * Vt) * dt + Vt_pos ** 0.5 * dB1 # log return increment
            
            Vt = Vt_pos + kappa * (theta - Vt_pos)* dt + sigma_V * Vt_pos ** 0.5 * dB2  # variance process
            
        returns[n] = log_ret
        
    return returns 

In [3]:
def h_tilde(kappa, h):
    "h_tidle = (1 - exp(-kappa * h)) / kappa"
    return (1 - np.exp(-kappa * h)) / kappa

def lagm_cov_est(ret, m):
    
    N = len(ret)
    Ybar = np.mean(ret)
    
    cov = np.sum((ret[:N-m] - Ybar) * (ret[m:] - Ybar)) / (N - m)
    return cov

def estimate_kappa(ret, h, lag_M=10, verbose=False):
    N    = len(ret)
    lag1 = lagm_cov_est(ret, 1)

    if verbose:
        print(f"  lag1 = {lag1:+.6e}")

    kappa_sum   = 0.0
    valid_count = 0

    for m in range(2, lag_M + 1):
        lagm = lagm_cov_est(ret, m)

        if lagm == 0 or np.sign(lag1) != np.sign(lagm):
            continue

        ratio = lag1 / lagm
        if ratio <= 0 or not np.isfinite(np.log(ratio)):
            continue

        k_est = np.log(ratio) / ((m-1) * h)
        if not np.isfinite(k_est) or k_est <= 0:
            continue

        if verbose:
            print(f"  m={m:>3d}  lagm={lagm:+.4e}  ratio={ratio:.4f}  k_est={k_est:.4f}")

        kappa_sum   += k_est
        valid_count += 1

    if valid_count == 0:
        if verbose:
            print("  No valid lag ratios found — returning 0")
        return 0.0

    kappa = kappa_sum / valid_count

    if verbose:
        print(f"  κ̂ = {kappa:.6f}  ({valid_count}/{lag_M-1} valid terms)")

    return kappa

def estimate_moments(ret):
    n = len(ret)
    Ybar = np.mean(ret)
    mean = Ybar
    var = np.var(ret, ddof = 1)
    lag1 = lagm_cov_est(ret, 1) 
    lag2 = lagm_cov_est(ret, 2)

    y2 = ret[:-1]**2
    y2d = y2 - np.mean(y2)
    yn1d = ret[1:] - np.mean(ret[1:])
    cross = np.mean(y2d * yn1d)
    
    return np.array([mean, var, lag1, lag2, cross])

def moments(mu, theta, kappa, sigma_V, rho, h):
    """Returns first five moments of the log returns of the Heston model."""
    ht = h_tilde(kappa, h)
    ek = np.exp(-kappa * h)
    
    # First moment (mean), follows from Theorem 2.1
    
    mean = h * (mu  - 0.5 * theta)
    
    # Second moment (variance), follows from Theorem 2.2
    
    var = theta * (h - ht) * ( (sigma_V**2)/ (4 * (kappa ** 2)) 
                              - (rho * sigma_V) / kappa) + theta * h
    
    # Third moment (lag-1 autocovariance), follows from Theorem 2.3
    
    lag1 = (((ht ** 2) * theta * (sigma_V)) / 2 ) * ( (sigma_V) / (4 * kappa) - rho)
    
    # Fourth moment (lag-2 autocovariance), follows from Theorem 2.4
    
    lag2 = ek * lag1
    
    
    # Fifth moment (Cross variance bewteen squared retursn and future returns),
    # follows from Theorem 2.5
    
    term1 = ((theta * (sigma_V ** 4) / ( 8 * kappa **3))) * ht * (h * ek - ht)
    
    term2 = ht**2 * ((theta * sigma_V ** 2) / (4 * kappa) * mu * h
                     - (theta ** 2 * sigma_V ** 2) / (8 * kappa) * h
                     - (theta * sigma_V ** 2 ) / (4 * kappa))
    
    term3  = (-rho * sigma_V / 2 * ht) * (
        (3 * sigma_V**2 / (2 * kappa**2) - 2 * rho * sigma_V / kappa)
        * theta * (h * ek - ht)
        + (2 * mu * theta - theta**2) * h * ht
)
    
    cross = term1 + term2 - term3
    
    return mean, var, lag1, lag2, cross

In [7]:
def estimate_heston_from_returns(ret, h, verbose=False, fallback=True):

    samp                                    = estimate_moments(ret)
    mean_e, var_e, lag1_e, lag2_e, cross_e = samp

    kappa = estimate_kappa(ret, h, lag_M=10, verbose = verbose)
    if not np.isfinite(kappa) or kappa <= 0:
        raise ValueError(f"Invalid kappa = {kappa}")

    # Cap kappa at economically plausible range
    kappa = np.clip(kappa, 0.01, 20.0)

    ht = (1 - np.exp(-kappa*h)) / kappa
    ek = np.exp(-kappa*h)
    dh = h*ek - ht

    theta = var_e/h - (2*(h-ht)/(h*kappa*ht**2))*lag1_e
    if not np.isfinite(theta) or theta <= 0:
        theta = var_e/h   # fallback
    theta = np.clip(theta, 1e-6, 2.0)   # cap at 141% annual vol

    mu = mean_e/h + theta/2

    num      = (4*kappa*mean_e + (8*dh/(theta*ht**3))*lag1_e
                - 2*kappa*cross_e/lag1_e)
    den      = theta*ht**2/(2*lag1_e) - dh/(kappa*ht)
    sigma_sq = num/den if abs(den) > 1e-14 else -1

    if not np.isfinite(sigma_sq) or sigma_sq <= 0:
        if fallback:
            var_y2   = np.var(ret**2, ddof=1)
            sigma_sq = max(2*kappa*var_y2/(theta**2*h**2), 1e-8)
        else:
            raise ValueError(f"sigma_sq = {sigma_sq:.4e} negative")

    # Cap sigma_V at economically plausible range
    sigma_V = np.clip(sigma_sq**0.5, 1e-4, 2.0)

    rho = sigma_V/(4*kappa) - (2/(theta*sigma_V*ht**2))*lag1_e
    if not np.isfinite(rho):
        if fallback:
            rho = np.corrcoef(np.abs(ret[:-1]), -ret[1:])[0, 1]
    rho = np.clip(rho, -0.999, 0.999)

    if verbose:
        print(f"  κ̂    = {kappa:.4f}")
        print(f"  θ̂    = {theta:.4f}")
        print(f"  μ̂    = {mu:.4f}")
        print(f"  σ̂_V  = {sigma_V:.6f}")
        print(f"  ρ̂    = {rho:.6f}")

    return {'mu': mu, 'theta': theta, 'kappa': kappa,
            'sigma_V': sigma_V, 'rho': rho}

In [ ]:
# Annualised parameters 
TRUE_financial = dict(
    mu      = 0.05,    # 5% annual drift
    theta   = 0.04,    # long-run variance → 20% annual vol
    kappa   = 2.0,     # mean reversion — half-life ≈ 4 months
    sigma_V = 0.3,     # vol of vol
    rho     = -0.7     # leverage effect
)

# Observation frequencies
h_daily   = 1  / 252   # ~0.00397 years per step
h_weekly  = 5  / 252   # ~0.01984 years per step  
h_monthly = 21 / 252   # ~0.08333 years per step
h_quarterly = 1/4
h_biannually = 1/2
h_annually = 1
h_2 = 2
h_4 = 4

# Expected signal at each frequency
for name, h in [('Daily', h_daily), ('Weekly', h_weekly), 
                ('Monthly', h_monthly), ('Quartely', h_quarterly),
                ('Bianually', h_biannually), ('Yearly', h_annually),
                ('Every Second Year', h_2), ('Every Fourth Year', h_4)]:
    ht    = (1 - np.exp(-TRUE_financial['kappa']*h)) / TRUE_financial['kappa']
    lag1  = (ht**2 * TRUE_financial['theta'] * TRUE_financial['sigma_V'] / 2) * \
            (TRUE_financial['sigma_V'] / (4*TRUE_financial['kappa']) - TRUE_financial['rho'])
    var   = TRUE_financial['theta'] * h
    snr   = abs(lag1) / (var / np.sqrt(400_000))
    ek    = np.exp(-TRUE_financial['kappa']*h)
    print(f"{name:<10s}  h={h:.5f}  e^(-κh)={ek:.4f}  "
          f"lag1={lag1:.4e}  SNR={snr:.2f}")

Daily       h=0.00397  e^(-κh)=0.9921  lag1=6.9130e-08  SNR=0.28
Weekly      h=0.01984  e^(-κh)=0.9611  lag1=1.6745e-06  SNR=1.33
Monthly     h=0.08333  e^(-κh)=0.8465  lag1=2.6072e-05  SNR=4.95
Quartely    h=0.25000  e^(-κh)=0.6065  lag1=1.7127e-04  SNR=10.83
Bianually   h=0.50000  e^(-κh)=0.3679  lag1=4.4203e-04  SNR=13.98
Yearly      h=1.00000  e^(-κh)=0.1353  lag1=8.2708e-04  SNR=13.08
Every Second Year  h=2.00000  e^(-κh)=0.0183  lag1=1.0661e-03  SNR=8.43
Every Fourth Year  h=4.00000  e^(-κh)=0.0003  lag1=1.1055e-03  SNR=4.37


In [17]:
def experiment_frequency(TRUE, V0, h_values, h_names, n_sub_dict,
                          n_rep=50, N=int(400_000)):
    """
    Replicates Table 3 of the paper but with daily/weekly/monthly
    observation frequencies instead of h = 0.5, 1, 2, 4.

    For each frequency:
      1. Simulate N observations at that frequency
      2. Estimate parameters
      3. Report mean, std, bias across n_rep replications

    Parameters
    ----------
    TRUE      : dict of true annualised parameters
    V0        : initial variance
    h_values  : list of h values  e.g. [1/252, 5/252, 21/252]
    h_names   : list of names     e.g. ['Daily', 'Weekly', 'Monthly']
    n_sub_dict: dict mapping h to n_sub  — finer sub-steps for larger h
    n_rep     : replications per frequency
    N         : observations per replication
    """
    keys        = ['mu', 'theta', 'kappa', 'sigma_V', 'rho']
    all_results = []

    for h, name in zip(h_values, h_names):
        n_sub  = n_sub_dict.get(h, 20)
        print(f"\n{'='*55}")
        print(f"Frequency: {name}  (h = {h:.5f},  n_sub = {n_sub})")
        print(f"  Total span : {N*h:.1f} years")
        print(f"  e^(-κh)    : {np.exp(-TRUE['kappa']*h):.6f}")
        print(f"  Feller     : {'✓' if 2*TRUE['kappa']*TRUE['theta'] >= TRUE['sigma_V']**2 else '✗'}")
        print(f"{'='*55}")

        estimates = {k: [] for k in keys}
        failed    = 0

        for r in range(n_rep):
            ret = heston_returns(V0, N,
                                 TRUE['mu'], TRUE['theta'], TRUE['kappa'],
                                 TRUE['sigma_V'], TRUE['rho'], h, n_sub)
            try:
                est = estimate_heston_from_returns(ret, h)
                for k in keys:
                    estimates[k].append(est[k])
            except Exception as ex:
                failed += 1

            if (r+1) % 10 == 0:
                print(f"  {r+1}/{n_rep}  failed: {failed}")

        row = {'name': name, 'h': h, 'N': N, 'failed': failed,
               'span': N*h, 'ek': np.exp(-TRUE['kappa']*h)}
        for k in keys:
            vals       = np.array(estimates[k])
            row[f'mean_{k}']  = vals.mean()  if len(vals)>0 else np.nan
            row[f'std_{k}']   = vals.std()   if len(vals)>0 else np.nan
            row[f'bias_{k}']  = vals.mean() - TRUE[k] if len(vals)>0 else np.nan
            row[f'rmse_{k}']  = np.sqrt((vals.mean()-TRUE[k])**2 + vals.var()) \
                                 if len(vals)>0 else np.nan
        all_results.append(row)

    # ── Mean ± Std table ───────────────────────────────────────────────────
    print(f"\n\n{'Frequency':<12s}  {'h':>8s}  " +
          "  ".join(f"{'mean('+k+')':>12s}" for k in keys))
    print("-" * 90)
    for row in all_results:
        print(f"  {row['name']:<10s}  {row['h']:8.5f}  " +
              "  ".join(f"{row['mean_'+k]:12.4f}" for k in keys))

    print(f"\n{'Frequency':<12s}  {'h':>8s}  " +
          "  ".join(f"{'std('+k+')':>12s}" for k in keys) +
          f"  {'Failed':>7s}")
    print("-" * 100)
    for row in all_results:
        print(f"  {row['name']:<10s}  {row['h']:8.5f}  " +
              "  ".join(f"{row['std_'+k]:12.4f}" for k in keys) +
              f"  {row['failed']:>7d}")

    # ── Bias table ─────────────────────────────────────────────────────────
    print(f"\n{'Frequency':<12s}  {'h':>8s}  " +
          "  ".join(f"{'bias('+k+')':>12s}" for k in keys))
    print("-" * 90)
    for row in all_results:
        print(f"  {row['name']:<10s}  {row['h']:8.5f}  " +
              "  ".join(f"{row['bias_'+k]:12.4f}" for k in keys))

    # ── SNR diagnostic ─────────────────────────────────────────────────────
    print(f"\nSignal-to-noise ratio for lag-1 autocovariance:")
    print(f"{'Frequency':<12s}  {'h':>8s}  {'lag1':>12s}  "
          f"{'noise':>12s}  {'SNR':>8s}  {'e^(-κh)':>10s}")
    print("-" * 68)
    for h, name in zip(h_values, h_names):
        ht   = (1 - np.exp(-TRUE['kappa']*h)) / TRUE['kappa']
        lag1 = (ht**2*TRUE['theta']*TRUE['sigma_V']/2) * \
               (TRUE['sigma_V']/(4*TRUE['kappa']) - TRUE['rho'])
        var  = TRUE['theta'] * h
        snr  = abs(lag1) / (var / np.sqrt(N))
        ek   = np.exp(-TRUE['kappa']*h)
        print(f"  {name:<10s}  {h:8.5f}  {lag1:12.4e}  "
              f"{var/np.sqrt(N):12.4e}  {snr:8.2f}  {ek:10.6f}")

    # ── LaTeX table ────────────────────────────────────────────────────────
    print(f"\n\nLaTeX table:\n")
    print(r"\begin{table}[h]")
    print(r"\centering")
    print(r"\caption{Effect of Observation Frequency on Estimation Accuracy}")
    print(r"\label{tab:frequency}")
    print(r"\begin{threeparttable}")
    print(r"\begin{tabular}{llc" + "c"*len(keys) + "}")
    print(r"\toprule")
    print(r"Frequency & $h$ & $e^{-\kappa h}$ & " +
          " & ".join([r"$\hat\mu$", r"$\hat\theta$",
                      r"$\hat\kappa$", r"$\hat\sigma_v$",
                      r"$\hat\rho$"]) + r" \\")
    print(r"\midrule")
    for row in all_results:
        vals = " & ".join(
            f"{row['mean_'+k]:.3f}$\\pm${row['std_'+k]:.3f}"
            for k in keys
        )
        print(f"  {row['name']} & ${row['h']:.5f}$ & "
              f"${row['ek']:.4f}$ & {vals} \\\\")
    print(r"\bottomrule")
    print(r"\end{tabular}")
    print(r"\begin{tablenotes}")
    print(r"\small")
    print(r"\item \textit{Notes:} Each cell reports mean $\pm$ std across "
          + str(n_rep) + r" replications of $N=" + f"{N:,}" +
          r"$ observations. All parameters are annualised: " +
          r"$\mu=" + str(TRUE['mu']) +
          r"$, $\kappa=" + str(TRUE['kappa']) +
          r"$, $\theta=" + str(TRUE['theta']) +
          r"$, $\sigma_v=" + str(TRUE['sigma_V']) +
          r"$, $\rho=" + str(TRUE['rho']) + r"$.")
    print(r"The column $e^{-\kappa h}$ measures the lag ratio signal strength ---")
    print(r"values closer to 0 indicate stronger identification of $\kappa$.")
    print(r"\end{tablenotes}")
    print(r"\end{threeparttable}")
    print(r"\end{table}")

    return all_results


# ── Run ────────────────────────────────────────────────────────────────────────
TRUE  = dict(mu=0.05, theta=0.04, kappa=2.0, sigma_V=0.3, rho=-0.7)
V0    = TRUE['theta']
n_sub = int(20)

h_daily   = 1  / 252
h_weekly  = 5  / 252
h_monthly = 21 / 252
h_quarterly = 1/4
h_biannually = 1/2
h_annually = 1
h_2 = 2
h_4 = 4

h_values = [h_daily,   h_weekly,   h_monthly, h_quarterly, h_biannually, h_annually, h_2, h_4]
h_names  = ['Daily',   'Weekly',   'Monthly', 'Quarterly', 'Biannually', 
            'Yearly', 'Every second year', 'Every fourth year']

# Sub-steps: paper uses n_sub=20 for h=1
# Scale n_sub inversely with h to keep dt ≈ 1/5040 constant

dt_target = 0.05
n_sub_dict = {h: max(1, int(np.ceil(h / dt_target))) for h in h_values}

results = experiment_frequency(
    TRUE       = TRUE,
    V0         = V0,
    h_values   = h_values,
    h_names    = h_names,
    n_sub_dict = n_sub_dict,
    n_rep      = 50,
    N          = int(400_000)
)



Frequency: Daily  (h = 0.00397,  n_sub = 1)
  Total span : 1587.3 years
  e^(-κh)    : 0.992095
  Feller     : ✓
  10/50  failed: 1
  20/50  failed: 2
  30/50  failed: 3
  40/50  failed: 5
  50/50  failed: 8

Frequency: Weekly  (h = 0.01984,  n_sub = 1)
  Total span : 7936.5 years
  e^(-κh)    : 0.961095
  Feller     : ✓
  10/50  failed: 2
  20/50  failed: 4
  30/50  failed: 5
  40/50  failed: 7
  50/50  failed: 7

Frequency: Monthly  (h = 0.08333,  n_sub = 2)
  Total span : 33333.3 years
  e^(-κh)    : 0.846482
  Feller     : ✓
  10/50  failed: 0
  20/50  failed: 0
  30/50  failed: 0
  40/50  failed: 0
  50/50  failed: 0

Frequency: Quarterly  (h = 0.25000,  n_sub = 5)
  Total span : 100000.0 years
  e^(-κh)    : 0.606531
  Feller     : ✓
  10/50  failed: 0
  20/50  failed: 0
  30/50  failed: 0
  40/50  failed: 0
  50/50  failed: 0

Frequency: Biannually  (h = 0.50000,  n_sub = 10)
  Total span : 200000.0 years
  e^(-κh)    : 0.367879
  Feller     : ✓
  10/50  failed: 0
  20/50  fail